# Notebook 06 — Exposure Characterisation, EDA & GeoAI
**GeoAI for Climate, Environment and Health** | 25 June 2026

---

## Learning objectives
- Join the participant-day timeline to the exposure grid by location and date
- Summarise daily exposures into trimester-level participant summaries
- Produce maps including participant migration paths
- Query a local DuckDB database with SQL
- Use a natural-language AI interface to query the database — and always check the generated SQL

**Time:** ~30 minutes

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import duckdb
import sys, pathlib, os

def _find_workshop_dir():
    here = pathlib.Path().resolve()
    for candidate in (here, here.parent, here.parent.parent):
        if (candidate / 'data').exists():
            return candidate
    return here

WORKSHOP_DIR = _find_workshop_dir()  # works whether run flat or from a notebooks/ subfolder
DATA_DIR = WORKSHOP_DIR / 'data'
sys.path.insert(0, str(WORKSHOP_DIR))
from shared_utils import find_nearest_grid_cells, assign_trimester, compute_trimester_summaries

participants = pd.read_csv(DATA_DIR / 'participants.csv',                        dtype={'PID': str})
visits       = pd.read_csv(DATA_DIR / 'visits.csv',                              dtype={'PID': str})
villages     = pd.read_csv(DATA_DIR / 'villages.csv')
preg_days    = pd.read_csv(DATA_DIR / 'participant_exposure_days.csv',           dtype={'PID': str})
exposure_grid = pd.read_csv(DATA_DIR / 'exposure_grid.csv', parse_dates=['date'])

print('Loaded all datasets.')

---
## Part A — Join  *(~10 min)*

### ✅ Worked example — Attach environmental values to every participant-day

In [ ]:
preg_days['date'] = pd.to_datetime(preg_days['date'])

# Step 1: for each unique village location, find the nearest grid cell.
unique_locs = preg_days[['latitude', 'longitude']].drop_duplicates().copy()
grid_ref = exposure_grid[['grid_id', 'lat', 'lon']].drop_duplicates('grid_id')
unique_locs = find_nearest_grid_cells(unique_locs, grid_ref)
print(f'Unique village locations: {len(unique_locs)}')

# Step 2: attach grid_id to every participant-day row.
preg_days = preg_days.merge(unique_locs[['latitude', 'longitude', 'grid_id']],
                             on=['latitude', 'longitude'], how='left')

# Step 3: merge on (grid_id, date) to attach environmental values.
env_cols = ['grid_id', 'date', 'temp_mean_c', 'temp_max_c', 'pm25_ugm3', 'ndvi', 'precipitation_mm', 'rh_mean']
preg_days_env = preg_days.merge(exposure_grid[env_cols], on=['grid_id', 'date'], how='left')

print(f'Joined table: {len(preg_days_env):,} rows, {preg_days_env.shape[1]} columns')
print(f'Missing temp_mean_c: {preg_days_env["temp_mean_c"].isna().sum()}')
preg_days_env.head()

---
## 🧪 Exercise 1 — Verify join completeness  *(2 points)*

Write a single assertion (or a print statement) that confirms **no** rows in `preg_days_env` have a missing `temp_mean_c` value. Store the count of missing values in `n_missing_temp` (should be 0).

In [ ]:
# Hint: preg_days_env['temp_mean_c'].isna().sum() counts the missing values.
# Store that count in n_missing_temp.

n_missing_temp = preg_days_env['temp_mean_c'].isna().sum()

print(f'Missing temp_mean_c values: {n_missing_temp}')
if n_missing_temp == 0:
    print('✅ Join is complete — every participant-day has an environmental value.')
else:
    print(f'⚠️  {n_missing_temp} rows have no temperature value — check the grid date range.')


---
### ✅ Knowledge Check — Verifying join completeness

*No coding here — just check your understanding before moving on.*

**Q1. Why bother checking for missing values *after* a join, instead of assuming the join worked because it ran without errors?**

- **A.** A join can run successfully but still produce missing values for rows that didn't find a match — checking for this is basic data-quality practice
- **B.** Joins never produce missing values in pandas
- **C.** This check is only relevant for SQL, not pandas
- **D.** Missing values after a join always mean the original data was corrupted

<details>
<summary>Show answer</summary>

**Answer: A** — A left join (or merge) will silently fill unmatched rows with NaN rather than raising an error — checking for missing values afterwards is how you catch incomplete joins before they bias your results.

</details>



In [ ]:
# Tests hidden — run your solution above first
pass

---
## Part B — Characterisation  *(~10 min)*

### ✅ Worked example — Trimester summaries

In [ ]:
preg_days_env['trimester'] = assign_trimester(preg_days_env['gestational_day'])

trimester_summaries = compute_trimester_summaries(preg_days_env)
print(f'Trimester summaries: {trimester_summaries.shape}')

participants_with_exp = participants.merge(trimester_summaries, on='PID', how='left')
output_dir = DATA_DIR if os.access(DATA_DIR, os.W_OK) else pathlib.Path.home()
if output_dir != DATA_DIR:
    print(f'Note: {DATA_DIR} is read-only here (you may be running from the shared course '
          f'copy rather than your own). Saving to your home directory instead.')

output_path = output_dir / 'participants_with_exposures_notebook.csv'
participants_with_exp.to_csv(output_path, index=False)
print(f'Saved {len(participants_with_exp):,} rows to {output_path}')
participants_with_exp[['PID', 'migrated', 'outcome', 'temp_mean_t1', 'temp_mean_t3', 'pm25_mean_t3']].head()

---
## 🧪 Exercise 2 — Migrant vs non-migrant exposure shift  *(4 points)*

For **migrant** and **non-migrant** participants separately, compute the mean difference in temperature between trimester 1 and trimester 3 (`temp_mean_t3 - temp_mean_t1`).

Store results in `migrant_temp_delta` and `nonmigrant_temp_delta` (both floats).

**What to observe:** Compare the sign and magnitude of the delta for each group. Migrants moved to a *different district* mid-pregnancy, so their T1 and T3 locations have different climate baselines — which can either amplify or partially offset the seasonal trend depending on the direction of movement. Non-migrants experience the full seasonal cycle in one place. After running the cell, discuss with your neighbour: does the result make sense given that ~30% of participants moved to a different district?

In [ ]:
# Hint: for each group (migrants / non-migrants), get each participant's
# mean T1 temperature and mean T3 temperature, then average (T3 - T1)
# across the participants in that group. Migrants/non-migrants can be
# identified from participants['migrated'].

participants_with_exp['temp_delta_t3_t1'] = participants_with_exp['temp_mean_t3'] - participants_with_exp['temp_mean_t1']
migrant_temp_delta     = float(participants_with_exp[participants_with_exp['migrated'] == True]['temp_delta_t3_t1'].mean())
nonmigrant_temp_delta  = float(participants_with_exp[participants_with_exp['migrated'] == False]['temp_delta_t3_t1'].mean())

print(f'Mean (T3 - T1) temperature delta:')
print(f'  Migrants:     {migrant_temp_delta:+.2f} °C')
print(f'  Non-migrants: {nonmigrant_temp_delta:+.2f} °C')
print(f'  Migrants show larger absolute shift: {abs(migrant_temp_delta) > abs(nonmigrant_temp_delta)}')


---
### ✅ Knowledge Check — Migration and exposure misclassification

*No coding here — just check your understanding before moving on.*

**Q1. If you ignored migration status and just used participants' *first recorded* village for their entire pregnancy, what would happen to the exposure estimate for migrants?**

- **A.** It would still be correct since location rarely affects temperature
- **B.** It would misrepresent part of their pregnancy, since they actually experienced a different climate after moving — this is a form of exposure misclassification
- **C.** It would only affect their age calculation, not exposure
- **D.** Migrants would automatically be excluded from the dataset

<details>
<summary>Show answer</summary>

**Answer: B** — Using a single static location for someone who moved mixes up their true time-varying exposure — a classic exposure misclassification problem in environmental epidemiology.

</details>

**Q2. Why compare the temperature delta (T3 − T1) between migrants and non-migrants specifically, rather than just looking at the whole cohort's average delta?**

- **A.** There's no reason — it's the same information either way
- **B.** Migration could be a source of effect modification or confounding: migrants' exposure pattern may systematically differ from non-migrants', which would be hidden by a single cohort-wide average
- **C.** Migrants are always excluded from such comparisons
- **D.** Non-migrants never have a temperature delta to compare

<details>
<summary>Show answer</summary>

**Answer: B** — Splitting by migration status helps reveal whether movement is associated with a meaningfully different exposure trajectory — pooling everyone together could mask that pattern.

</details>



In [ ]:
# Tests hidden — run your solution above first
pass

---
## Part C — EDA & Maps  *(~10 min)*

### ✅ Worked example — Four exploratory plots

In [ ]:
# Plot 1: Boxplot of third-trimester temperature by outcome.
outcomes_order = ['live_birth_term', 'low_birth_weight', 'preterm_birth', 'stillbirth']
outcome_colours = {'live_birth_term': '#2a9d8f', 'low_birth_weight': '#e9c46a',
                   'preterm_birth': '#f4a261', 'stillbirth': '#e63946'}

fig, ax = plt.subplots(figsize=(9, 5))
data_for_box = [participants_with_exp[participants_with_exp['outcome'] == o]['temp_mean_t3'].dropna()
                for o in outcomes_order]
bp = ax.boxplot(data_for_box, labels=outcomes_order, patch_artist=True, notch=False)
for patch, o in zip(bp['boxes'], outcomes_order):
    patch.set_facecolor(outcome_colours[o])
    patch.set_alpha(0.7)
ax.set_xlabel('Outcome')
ax.set_ylabel('Mean temperature — trimester 3 (°C)')
ax.set_title('Third-trimester heat exposure by birth outcome')
plt.tight_layout()
plt.show()

In [ ]:
# Plot 2: PM2.5 vs birthweight scatter (live births only).
live = participants_with_exp[participants_with_exp['birthweight_g'].notna()].copy()

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(live['pm25_mean_t3'], live['birthweight_g'], alpha=0.6, edgecolors='white', s=50, color='#457b9d')

m, b = np.polyfit(live['pm25_mean_t3'].dropna(), live['birthweight_g'].dropna(), 1)
x_range = np.linspace(live['pm25_mean_t3'].min(), live['pm25_mean_t3'].max(), 100)
ax.plot(x_range, m * x_range + b, color='#e63946', linewidth=2, label=f'Trend (slope={m:.0f} g/µg/m³)')
ax.set_xlabel('Mean PM2.5 — trimester 3 (µg/m³)')
ax.set_ylabel('Birthweight (g)')
ax.set_title('PM2.5 exposure vs birthweight — live births')
ax.legend()
ax.axhline(2500, color='grey', linestyle='--', alpha=0.5, label='Low birthweight threshold')
plt.tight_layout()
plt.show()

In [ ]:
# Plot 3: Delivery locations coloured by outcome.
delivery_visits = visits[visits['visit_type'] == 'delivery'].merge(
    participants_with_exp[['PID', 'outcome']], on='PID', how='left')

fig, ax = plt.subplots(figsize=(8, 6))
for outcome, colour in outcome_colours.items():
    sub = delivery_visits[delivery_visits['outcome'] == outcome]
    ax.scatter(sub['longitude'], sub['latitude'], c=colour, label=outcome,
               s=60, alpha=0.8, edgecolors='white', linewidths=0.5)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('Delivery locations by birth outcome')
ax.legend(fontsize=8)
# Add a real-world OpenStreetMap basemap for context. This needs internet
# access to fetch map tiles — if that fails (e.g. no connectivity at the
# venue), we fall back to the plain scatter map above instead of crashing.
try:
    import contextily as cx
    cx.add_basemap(ax, crs='EPSG:4326', source=cx.providers.CartoDB.Positron, zoom=6)
except Exception as e:
    print(f'(Basemap unavailable — showing plain scatter map instead: {e})')

plt.tight_layout()
plt.show()

In [ ]:
# Plot 4: Migration paths — all 3 visits per migrant participant, connected by lines.
migrant_pids = participants_with_exp[participants_with_exp['migrated'] == True]['PID'].tolist()
migrant_visits = visits[visits['PID'].isin(migrant_pids)].sort_values(['PID', 'visit_date'])

fig, ax = plt.subplots(figsize=(9, 7))
cmap_migrant = cm.get_cmap('tab20', len(migrant_pids))

for i, pid in enumerate(migrant_pids):
    pv = migrant_visits[migrant_visits['PID'] == pid]
    colour = cmap_migrant(i)
    ax.plot(pv['longitude'], pv['latitude'], '-o', color=colour, alpha=0.7, markersize=5, linewidth=1.2)
    # Mark the delivery point with a star.
    deliv = pv[pv['visit_type'] == 'delivery']
    ax.scatter(deliv['longitude'], deliv['latitude'], marker='*', s=120, color=colour, zorder=5)

ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title(f'Migration paths during pregnancy (n={len(migrant_pids)} migrants)  ★ = delivery')
# Add a real-world OpenStreetMap basemap for context. This needs internet
# access to fetch map tiles — if that fails (e.g. no connectivity at the
# venue), we fall back to the plain scatter map above instead of crashing.
try:
    import contextily as cx
    cx.add_basemap(ax, crs='EPSG:4326', source=cx.providers.CartoDB.Positron, zoom=6)
except Exception as e:
    print(f'(Basemap unavailable — showing plain scatter map instead: {e})')

plt.tight_layout()
plt.show()

---
## 🧪 Exercise 3 — Interpret the PM2.5 scatter  *(3 points)*

In the cell below, write 2–3 sentences describing what the PM2.5 vs birthweight scatter plot suggests. Consider: what is the direction of the trend? Is it strong or weak? What would you want to check before drawing any conclusion?

*(This is a free-text cell — there is no single correct answer. Your facilitator will read and mark it.)*

*(Write your 2–3 sentences here)*

---
## Part D — DuckDB & GeoAI  *(~10–15 min)*

### ✅ Worked example — Connect to the database and run guided SQL queries

In [ ]:
DB_PATH = str(DATA_DIR / 'exposures.duckdb')
con = duckdb.connect(DB_PATH, read_only=True)

# Check what tables are available.
print('Tables:', con.execute("SHOW TABLES").fetchdf()['name'].tolist())

In [ ]:
# Guided query 1: count participants with hot third trimesters.
q1 = con.execute("""
    SELECT COUNT(*) AS n_hot_participants
    FROM participant_summary
    WHERE temp_mean_t3 > 28
""").fetchdf()
print('Participants with mean T3 temp > 28°C:')
q1

In [ ]:
# Guided query 2: average PM2.5 by outcome.
q2 = con.execute("""
    SELECT outcome,
           COUNT(*) AS n,
           ROUND(AVG(pm25_mean_t3), 2) AS avg_pm25_t3
    FROM participant_summary
    WHERE outcome IS NOT NULL
    GROUP BY outcome
    ORDER BY avg_pm25_t3 DESC
""").fetchdf()
print('Average PM2.5 in trimester 3 by outcome:')
q2

In [ ]:
# Guided query 3 (spatial — only runs if the spatial extension loaded).
try:
    # Reuse the existing connection — LOAD doesn't write to the database
    # file, so it works fine on the read-only connection opened above.
    # (Opening a second connection to the same file with a different
    # read-only/read-write mode while the first is still open raises a
    # Connection Error in DuckDB.)
    con_rw = con
    con_rw.execute("LOAD spatial")
    target_village = 'Riverside'
    target = villages[villages['village'] == target_village].iloc[0]
    q3 = con_rw.execute(f"""
        SELECT grid_id, lat, lon,
               ST_Distance(
                   ST_Point(lon, lat),
                   ST_Point({target['longitude']}, {target['latitude']})
               ) AS dist_deg
        FROM exposure_grid
        WHERE dist_deg < 0.3
        GROUP BY grid_id, lat, lon
        ORDER BY dist_deg
        LIMIT 10
    """).fetchdf()
    print(f'Grid cells within 0.3° of {target_village}:')
    print(q3)
except Exception as e:
    print(f'Spatial query skipped: {e}')

---
## 🧪 Exercise 4 — Write a SQL query  *(2 points)*

Write a SQL query that finds the **district with the highest mean third-trimester temperature** across all participants whose delivery village is in that district.

Hint: join `participant_summary` to `visits` (delivery visit) to `villages` to get the district.

Store the result DataFrame in `hottest_district_df`.

In [ ]:
# Hint: a starting skeleton —
# SELECT v.district, AVG(p.temp_mean_t3) AS mean_t3
# FROM participant_summary p
# JOIN visits ON ...      -- delivery visit per PID
# JOIN villages v ON ...  -- delivery village -> district
# GROUP BY v.district
# ORDER BY mean_t3 DESC
# Store the query result (a DataFrame) in hottest_district_df.

hottest_district_df = con.execute("""
    SELECT v.district,
           ROUND(AVG(ps.temp_mean_t3), 2) AS avg_temp_t3,
           COUNT(*) AS n_participants
    FROM participant_summary ps
    JOIN visits vst ON ps.PID = vst.PID AND vst.visit_type = 'delivery'
    JOIN villages v  ON vst.village = v.village
    WHERE ps.temp_mean_t3 IS NOT NULL
    GROUP BY v.district
    ORDER BY avg_temp_t3 DESC
""").fetchdf()

hottest_district_df


In [ ]:
# Tests hidden — run your solution above first
pass

---
### ✅ Knowledge Check — District-level aggregation for decision-making

*No coding here — just check your understanding before moving on.*

**Q1. What's the practical reason to aggregate exposure data to the *district* level (as the SQL query does) rather than only ever looking at individual participants?**

- **A.** District-level numbers are always more statistically significant
- **B.** Health policy and resource allocation (e.g. cooling centres, clinics) is usually planned at administrative levels like districts, so results need to be summarised that way to be actionable
- **C.** Individual-level data is never reliable
- **D.** Aggregation removes the need for any further analysis

<details>
<summary>Show answer</summary>

**Answer: B** — Administrative/health-system decisions are typically made at the district (or similar) level, so presenting findings at that level makes them directly usable by decision-makers.

</details>



---
## 📖 GeoAI — Natural language to SQL

The GeoAI interface below sends your question to a large language model (Claude Haiku) running on the PRECISE server. The model:

1. Receives a description of the database schema (table and column names only — no data)
2. Returns a **single SQL SELECT statement** — nothing else
3. That SQL is **shown to you** before the query runs so you can check it
4. The query is validated: only `SELECT` is allowed — `INSERT`, `UPDATE`, `DROP`, etc. are blocked

**Always check the SQL.** AI-generated SQL can misinterpret column names or join logic. Treat it as a first draft to verify, not a definitive answer.

If the server is unavailable, a demo mode runs instead — you will see example questions and their pre-written SQL responses.

---
### ✅ GeoAI interface

In [ ]:
import requests, json
import ipywidgets as widgets
from IPython.display import display, clear_output

# Server IP — connects to the PRECISE server over the workshop LAN.
GEOAI_URL = 'http://SERVER_IP:5000/api/workshop/geoai'

# Your JupyterHub username — sent with each question so the server can track
# your token budget (keeps everyone's exploring without anyone burning through
# the shared budget).
HUB_USERNAME = os.environ.get('JUPYTERHUB_USER', os.environ.get('USER', 'student'))

# Pre-written demo pairs (used when the server is unreachable).
DEMO_QA = [
    (
        'How many participants had a preterm birth?',
        "SELECT COUNT(*) AS n_preterm FROM participants WHERE outcome = 'preterm_birth'"
    ),
    (
        'Which village had the highest average PM2.5 in the third trimester?',
        """SELECT vst.village, ROUND(AVG(ps.pm25_mean_t3),2) AS avg_pm25
FROM participant_summary ps
JOIN visits vst ON ps.PID = vst.PID AND vst.visit_type = 'delivery'
GROUP BY vst.village ORDER BY avg_pm25 DESC LIMIT 1"""
    ),
    (
        'What is the average birthweight for term vs preterm births?',
        """SELECT outcome, ROUND(AVG(birthweight_g),0) AS avg_bw
FROM participants
WHERE birthweight_g IS NOT NULL
GROUP BY outcome ORDER BY avg_bw DESC"""
    ),
    (
        'How many participants migrated?',
        "SELECT COUNT(*) AS n_migrants FROM participants WHERE migrated = true"
    ),
    (
        'List districts ordered by average NDVI in trimester 2',
        """SELECT v.district, ROUND(AVG(ped.ndvi),3) AS avg_ndvi
FROM participant_exposure_days_with_env ped
JOIN visits vst ON ped.PID = vst.PID AND vst.visit_type = 'delivery'
JOIN villages v ON vst.village = v.village
WHERE ped.trimester = 2
GROUP BY v.district ORDER BY avg_ndvi DESC"""
    ),
]
demo_idx = [0]

def run_geoai(question: str):
    """Send question to GeoAI endpoint; fall back to demo mode on error."""
    try:
        resp = requests.post(GEOAI_URL, json={'question': question, 'username': HUB_USERNAME}, timeout=15)
        resp.raise_for_status()
        data = resp.json()
        if data.get('error'):
            return None, data['error']
        return data.get('sql'), None
    except Exception as e:
        # Demo mode fallback.
        i = demo_idx[0] % len(DEMO_QA)
        demo_idx[0] += 1
        _q, _sql = DEMO_QA[i]
        return _sql, f'[DEMO MODE — server unreachable ({e}).  Showing pre-written example for: "{_q}"]'

# Widgets.
question_label = widgets.HTML(
    "<div style='font-size:15px; font-weight:bold; margin-bottom:4px;'>"
    "&#x1F4AC; Type your question here, then click 'Ask GeoAI' (or press Enter):</div>"
)
question_box = widgets.Text(
    placeholder='e.g. How many participants migrated?',
    layout=widgets.Layout(width='70%', height='38px'),
    style={'description_width': 'initial'},
)
submit_btn   = widgets.Button(description='Ask GeoAI', button_style='primary',
                               layout=widgets.Layout(width='150px', height='38px'))
output_area  = widgets.Output()

def auto_plot(result_df):
    """Pick a sensible chart for the result, the same way Shmron AI (the
    catalogue chat assistant) auto-selects a chart type for query results —
    here using matplotlib since we're in a notebook, not a browser chart lib."""
    if result_df.empty or len(result_df) > 30:
        return  # too many rows for a readable auto-chart — table above is enough
    numeric_cols = result_df.select_dtypes(include='number').columns.tolist()
    other_cols   = [c for c in result_df.columns if c not in numeric_cols]
    if not numeric_cols:
        return  # nothing to plot

    fig, ax = plt.subplots(figsize=(7, 4))
    try:
        if len(other_cols) >= 1 and len(numeric_cols) == 1 and result_df[other_cols[0]].nunique() <= 30:
            # one category column + one numeric column -> bar chart
            cat_col, val_col = other_cols[0], numeric_cols[0]
            ax.bar(result_df[cat_col].astype(str), result_df[val_col], color='#457b9d')
            ax.set_xlabel(cat_col)
            ax.set_ylabel(val_col)
            plt.xticks(rotation=45, ha='right')
        elif len(numeric_cols) >= 2:
            # two or more numeric columns -> scatter of the first two
            x_col, y_col = numeric_cols[0], numeric_cols[1]
            ax.scatter(result_df[x_col], result_df[y_col], color='#e63946', alpha=0.8)
            ax.set_xlabel(x_col)
            ax.set_ylabel(y_col)
        else:
            plt.close(fig)
            return
        ax.set_title('Auto-generated chart for this result')
        plt.tight_layout()
        plt.show()
    except Exception:
        plt.close(fig)  # if anything about the data shape trips up plotting, just skip the chart


def on_submit(_):
    q = question_box.value.strip()
    if not q:
        return
    with output_area:
        clear_output()
        print(f'Question: {q}\n')
        sql, err = run_geoai(q)
        if err:
            print(f'Note: {err}\n')
        if sql:
            print('Generated SQL (always check this!):')
            print(sql)
            print()
            try:
                # Open a short-lived connection just for this query, rather than
                # relying on the long-lived 'con' from earlier in the notebook —
                # if you've already run the whole notebook (which closes 'con' in
                # the last cell), this still works.
                with duckdb.connect(DB_PATH, read_only=True) as qcon:
                    result_df = qcon.execute(sql).fetchdf()
                print(f'Result ({len(result_df)} rows):')
                display(result_df)
                auto_plot(result_df)
            except Exception as qerr:
                print(f'SQL execution error: {qerr}')
        else:
            print('No SQL returned.')

submit_btn.on_click(on_submit)
question_box.on_submit(on_submit)  # Enter key also submits
display(question_label, widgets.HBox([question_box, submit_btn]), output_area)

---
## 🧪 Final exercise — Write your own GeoAI questions  *(ungraded — reflection)*

In the **GeoAI box above** (the one you just used), run **two** questions of your own through the GeoAI interface. Then answer:
1. Did the generated SQL match what you intended?
2. Did the result match what you expected from the earlier plots?

*(Write your observations in the markdown cell below)*

**My question 1:**

**Generated SQL:** *(paste here)*

**Did it match my intent?** *(yes / no / partially — explain)*

---

**My question 2:**

**Generated SQL:** *(paste here)*

**Did it match my intent?** *(yes / no / partially — explain)*

---
## 🏁 Full course recap

Across all six notebooks you have built the complete chain:

| Step | Notebook | Output |
|---|---|---|
| Geocoded health data | 01 | Maps of facilities by type/district |
| Environmental grids | 02 | 2×2 spatial maps of climate variables |
| Spatial join (static) | 03 | Environmental values attached to points |
| Time-varying location | 04 | `participant_exposure_days.csv` |
| EE extraction | 05 | Understanding of satellite data sources |
| Exposure characterisation | 06 | Trimester summaries → plots → SQL → GeoAI |

The key insight: **a surveyor's expertise in spatial data — knowing where things are, how locations are recorded, and how coordinate systems work — is directly applicable to climate-health research.** The same nearest-neighbour join you wrote in Notebook 03 is the core of what large-scale epidemiological platforms use to link participant locations to environmental exposures.

The GeoAI layer shows where the field is heading: structured spatial databases queried through natural language, with the SQL always visible so the human expert can verify the machine's reasoning.

In [ ]:
con.close()
print('Database connection closed. Thank you for completing the workshop!')